# ICARUS: диагностика лазера
Укажи папку `laser_000_...` с CSV. Затем выполни ячейки по порядку. Нужны NumPy и Matplotlib.

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt

folder = Path("/Users/dmitrykiramov_1/Desktop/tst_proj/ICARUS_geometry_pulse_diagnostics/results/example_diagnostics/run_20260924T124708_1790254028854469Z_1752376437_0/laser_000_main")

def read_csv(path):
    with path.open() as f:
        lines = [line for line in f if line.strip() and not line.lstrip().startswith('#')]
    return np.atleast_1d(np.genfromtxt(lines, delimiter=',', names=True))

time = read_csv(folder / 'power_laser_time.csv')
snapshots = sorted(folder.glob('power_laser_theta_*.csv'))
print(f'Угловых снимков: {len(snapshots)}')


## Мощность по времени
Мощности усреднены по транспортному шагу, не по периоду записи файлов.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for key, label in [
    ('absorbed_power_w', 'Absorbed'),
    ('escaped_power_w', 'Escaped total'),
    ('toward_laser_power_w', 'Toward laser'),
    ('away_from_laser_power_w', 'Away from laser'),
]:
    ax.plot(time['time_s'] * 1e9, time[key], '.-', label=label)
ax.set(xlabel='Time [ns]', ylabel='Power [W]')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


## Мощность по углам
Измени `snapshot_index`: `0` — первый снимок, `-1` — последний. Значения в ваттах на бин, не Вт/ср. 0° — вперёд, 180° — к лазеру.

In [ ]:
snapshot_index = 0

path = snapshots[snapshot_index]
angle = read_csv(path)
metadata = dict(re.findall(r'([A-Za-z_]+)=([^\s]+)', path.read_text()))
print(path.name)
print('Transport status:', metadata.get('transport_status', '?'),
      '| partial:', metadata.get('partial', '?'),
      '| angular complete:', metadata.get('angular_complete', '?'),
      '| unbinned [W]:', metadata.get('unbinned_escaped_power_w', '?'))

edges = np.r_[angle['theta_left_deg'], angle['theta_right_deg'][-1]]
fig, ax = plt.subplots(figsize=(9, 4))
for key, label in [('power_total_w', 'Total'), ('power_p_w', 'p'), ('power_s_w', 's')]:
    ax.stairs(angle[key], edges, label=label)
ax.axvline(90, color='gray', linestyle='--', linewidth=1)
t_ns = float(metadata['time_s']) * 1e9
ax.set(xlabel='Angle relative to laser axis [deg]', ylabel='Power per bin [W]',
       xlim=(0, 180), title=f't = {t_ns:g} ns')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
